# Security Framing in the European Parliament — Pipeline

This notebook runs the complete pipeline from corpus segmentation to inductive topic cluster exploration and zero-shot classification, and robustness checks end-to-end. Checkpoints are written to Google Drive so one can safely disconnect and resume.

**Before you start:**
1. Runtime → Change runtime type → **A100** (if available).
2. Connect to **Google Drive** in Cell 2 to store intermediate and final outputs, set up under this link: https://drive.google.com/drive/folders/1oUIqaypIW2Cz_0lZ4P3UhrYXNlsLfdko?dmr=1&ec=wgc-drive-%5Bmodule%5D-goto and confirm that `corpus_ep_security_CLEAN.xlsx` is uploaded under `/data/`. (if not in `/data/` on Google Drive, then download them from **GitHub** and upload them).
3. Confirm that all necessary Python + R scripts via Cell 2 and 3 (if not in `/scripts/` on Google Drive, then download them from **GitHub** and upload them).
4. Run cells in order. Cells are checkpointed, so safe to interrupt.

**Cell overview:**

| Cell | Step |
|---|---|
| 4-7 | Translation (NLLB-200) |
| 8 | Segmentation |
| 9-11 | Topic clusters with BERTopic EN/ML |
| 12-14 | Paragraph-level classification (ML) with 13 and 14 as optional intermediate and final quick summaries |
| 15 | Context-window ML |
| 16 | Robustness check |


In [5]:
# ==============================================================================
# CELL 1 — Confirm GPU + install packages
# ==============================================================================
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('⚠  No GPU — go to Runtime → Change runtime type → A100 GPU')
else:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {name}  ({vram:.1f} GB VRAM)')
    print(f'  PyTorch: {torch.__version__}')

!pip install -q \
    "sentence-transformers" \
    "bertopic" \
    "umap-learn" "hdbscan" \
    "nltk" "sentencepiece" "openpyxl" \
    "tqdm"

print('Packages installed.')

Device: cuda
✓ GPU: NVIDIA A100-SXM4-40GB  (42.4 GB VRAM)
  PyTorch: 2.10.0+cu128
Packages installed.


In [19]:
# ==============================================================================
# CELL 2 — Mount Drive and set up working paths (if accessed from Drive)
# ==============================================================================

import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Project directory on Drive
PROJECT_DIR = '/content/drive/MyDrive/classification_pipeline_security_framing'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
SCRIPT_DIR  = os.path.join(PROJECT_DIR, 'scripts')
CKPT_DIR    = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(SCRIPT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

print(f'PROJECT_DIR: {PROJECT_DIR}')

# Checks the corpus is there; if not: drop-in
corpus = os.path.join(DATA_DIR, 'corpus_ep_security_CLEAN.xlsx')
if os.path.exists(corpus):
    import pandas as pd
    n = len(pd.read_excel(corpus))
    print(f'✓ Corpus found: {corpus}  ({n} speeches)')
else:
    print(f'⚠  Corpus not found at {corpus}')
    print('   Upload corpus_ep_security_CLEAN.xlsx to the data/ folder of')
    print('   your Drive project, then re-run this cell.')


Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/classification_pipeline_security_framing
✓ Corpus found: /content/drive/MyDrive/classification_pipeline_security_framing/data/corpus_ep_security_CLEAN.xlsx  (19859 speeches)


In [20]:
# ==============================================================================
# CELL 3 — Upload scripts from your local machine (or skip if already in Drive)
# ==============================================================================
from google.colab import files
import os, shutil

expected_scripts = [
    'frames.py',
    '00_translate.py',
    '00_translate_v3.py',
    '00_truncate_loops.py',
    '00_flag_for_retranslation.py',
    '01_segment_units.py',
    '02a_bertopic_english.py',
    '02b_bertopic_multilingual.py',
    '03_classify_frames_zeroshot.py',
    '04_robustness_checks.py',
]

missing = [f for f in expected_scripts
           if not os.path.exists(os.path.join(SCRIPT_DIR, f))]
if not missing:
    print('All scripts already present in Drive. Skipping upload.')
else:
    print(f'Missing in Drive: {missing}')
    print('Please select the Python/R scripts from your local disk:')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        dest = os.path.join(SCRIPT_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(content)
        print(f'  saved -> {dest}')

# Sanity check after upload
for f in expected_scripts:
    p = os.path.join(SCRIPT_DIR, f)
    mark = '✓' if os.path.exists(p) else '✗'
    print(f'  {mark}  {f}')


All scripts already present in Drive. Skipping upload.
  ✓  frames.py
  ✓  00_translate.py
  ✓  00_translate_v3.py
  ✓  00_truncate_loops.py
  ✓  00_flag_for_retranslation.py
  ✓  01_segment_units.py
  ✓  02a_bertopic_english.py
  ✓  02b_bertopic_multilingual.py
  ✓  03_classify_frames_zeroshot.py
  ✓  04_robustness_checks.py


In [5]:
# ==============================================================================
# CELL 4 — Translate non-English speeches
# ==============================================================================
# Writes corpus_ep_security_TRANSLATED.xlsx with a new `speech_translated`
# column; the original corpus is never modified. Safe to re-run: if the
# checkpoint CSV exists, only remaining speeches are translated.

!cd {SCRIPT_DIR} && python 00_translate.py \
    --input  {DATA_DIR}/corpus_ep_security_CLEAN.xlsx \
    --output {DATA_DIR}/corpus_ep_security_TRANSLATED.xlsx \
    --checkpoint_dir {CKPT_DIR}


Loading /content/drive/MyDrive/ep_security_framing/data/corpus_ep_security_CLEAN_2.xlsx...
  19859 speeches
  Resume: 18291 rows already translated
  To translate now: 1568 rows across 6 languages

Loading NLLB-200 on cuda (torch.float16)...
config.json: 100% 808/808 [00:00<00:00, 4.70MB/s]
tokenizer_config.json: 100% 564/564 [00:00<00:00, 3.53MB/s]
tokenizer.json: 100% 17.3M/17.3M [00:01<00:00, 10.2MB/s]
special_tokens_map.json: 3.55kB [00:00, 421kB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100% 5.48G/5.48G [00:17<00:00, 322MB/s]
Loading weights:  87% 880/1016 [00:01<00:00, 729.37it/s, Materializing param=model.encoder.layers.15.self_attn.out_proj.bias]
Loading weights: 100% 1016/1016 [00:01<00:00, 779.45it/s, Materializing param=model.shared.weight]

model.safetensors:   0% 0.00/5.48G [00:00<?, ?B/s]The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we wil

In [16]:
# ==============================================================================
# CELL 5 — Upon inspection w/ Claude, we see the model is looping in about
# 1,500/18,000 rows i.e. getting stuck on the same word. This code flags loops
# and re-translates these rows, this time with repetition penalty in place.
#
# Note: For a clean first-time run replace the translation script in Cell 3
# with 00_translate_v3.py. Given time constraints and long run times, we correct
# ex-post using flags.
# ==============================================================================

!python3 $SCRIPT_DIR/00_flag_for_retranslation.py \
    --translated  $DATA_DIR/corpus_ep_security_TRANSLATED.xlsx \
    --checkpoint  $CKPT_DIR/translations_checkpoint.csv \
    --out_checkpoint $CKPT_DIR/translations_checkpoint.csv

Loading translated corpus: /content/drive/MyDrive/ep_security_framing/data/corpus_ep_security_TRANSLATED.xlsx
  19859 rows total
Loading checkpoint: /content/drive/MyDrive/ep_security_framing/checkpoints/translations_checkpoint.csv
  19859 rows in checkpoint
Scoring translations for repetition...
  Flagged 509 rows for re-translation (rep_score>3 AND loop starts before 20% of text)
  Removed 509 rows from checkpoint (19859 -> 19350)
  Saved cleaned checkpoint -> /content/drive/MyDrive/ep_security_framing/checkpoints/translations_checkpoint.csv

Flagged rows by language:
language_detected
DE    169
FR     80
MT     75
PL     58
ES     37
IT     18
LV     10
RO     10
HR      7
NL      7
PT      5
ET      4
LT      4
SV      4
CS      4
HU      4
DA      4
FI      3
SL      2
SK      2
BS      1
GA      1

Done. Now run 00_translate_v3.py with the same arguments as before.
It will skip all clean rows and only re-translate the flagged ones.


In [19]:
# ==============================================================================
# CELL 6 — continuation
# ==============================================================================

!python3 $SCRIPT_DIR/00_translate_v3.py \
    --input      $DATA_DIR/corpus_ep_security_CLEAN.xlsx \
    --output     $DATA_DIR/corpus_ep_security_TRANSLATED_v3.xlsx \
    --checkpoint_dir $CKPT_DIR

Loading /content/drive/MyDrive/ep_security_framing/data/corpus_ep_security_CLEAN_2.xlsx...
  19859 speeches
  Resume: 19350 rows already translated
  To translate now: 509 rows across 22 languages

Loading NLLB-200 on cuda (torch.float16)...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 1016/1016 [00:01<00:00, 771.86it/s, Materializing param=model.shared.weight]
The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie mode

In [22]:
# ==============================================================================
# CELL 7 — for about 3% of total rows, the loop is not resolved.
# To avoid the overemphasis of looped words distorting the classification,
# we truncate at the start of the loop. This allows to preserve > 45% of speech
# per row, on average. This is a notable limitation but only affects 3% of rows.
# ==============================================================================

!python3 $SCRIPT_DIR/00_truncate_loops.py \
    --input  $DATA_DIR/corpus_ep_security_TRANSLATED_v3.xlsx \
    --output $DATA_DIR/corpus_ep_security_TRANSLATED_v3_clean.xlsx


Loading /content/drive/MyDrive/ep_security_framing/data/corpus_ep_security_TRANSLATED_v3.xlsx...
  19,859 rows
Scoring translations for repetition loops...
  scoring: 100% 19859/19859 [00:02<00:00, 7296.88it/s]
  Rows with loops: 810
Truncating looped translations...
  truncating: 100% 810/810 [00:00<00:00, 2802.49it/s]
  Truncated: 807 rows
  Unchanged (loop at pos 0 or undetected): 3

  Avg translated length as % of original speech: 44.8%

Writing output -> /content/drive/MyDrive/ep_security_framing/data/corpus_ep_security_TRANSLATED_v3_clean.xlsx
  translation_truncated=True:  807
  translation_truncated=False: 19,052
Done.


In [24]:
# ==============================================================================
# CELL 8 — Segment into speech / sentence / context-window
# ==============================================================================

!cd {SCRIPT_DIR} && python 01_segment_units.py \
    --input      {DATA_DIR}/corpus_ep_security_TRANSLATED_v3_clean.xlsx \
    --output_dir {DATA_DIR}


Ensuring NLTK punkt tokenisers...

Loading /content/drive/MyDrive/ep_security_framing/data/corpus_ep_security_TRANSLATED_v3_clean.xlsx...

--- Building ALL_ML from column 'speech_text' ---
  wrote  19859 speeches   -> /content/drive/MyDrive/ep_security_framing/data/speeches_ALL_ML.csv
  segment (ALL_ML): 100% 19859/19859 [00:16<00:00, 1178.42it/s]
  wrote 285450 sentences  -> /content/drive/MyDrive/ep_security_framing/data/sentences_ALL_ML.csv
  wrote 285450 windows    -> /content/drive/MyDrive/ep_security_framing/data/context_windows_ALL_ML.csv

--- Building ALL_EN from column 'speech_translated' ---
  wrote  19859 speeches   -> /content/drive/MyDrive/ep_security_framing/data/speeches_ALL_EN.csv
  segment (ALL_EN): 100% 19859/19859 [00:13<00:00, 1480.07it/s]
  wrote 236585 sentences  -> /content/drive/MyDrive/ep_security_framing/data/sentences_ALL_EN.csv
  wrote 236585 windows    -> /content/drive/MyDrive/ep_security_framing/data/context_windows_ALL_EN.csv

Segmentation complete.
NEXT

In [ ]:
# ==============================================================================
# CELL 9 -- Inductive exploration of prevalent topics using BERTopic
# on English-translated corpus
# ==============================================================================
# Embeddings are cached so reruns are very fast if only HDBSCAN/UMAP settings
# are changed. Run on SPEECH level for the purpose of this exercise
# Possibility to switch --unit to context_window for a finer-grained view.

!cd {SCRIPT_DIR} && python 02a_bertopic_english.py \
    --data_dir {DATA_DIR} --unit speech


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 16.1 MB/s eta 0:00:00
2026-04-24 17:08:52.052327: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-24 17:08:52.070536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777050532.094372   31551 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777050532.102575   31551 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777050532.123496   31551 computation_placer.cc:17

In [5]:
# ==============================================================================
# CELL 10 -- Inductive exploration of prevalent topics using BERTopic
# per language (multilingual encoder)
# ==============================================================================

!cd {SCRIPT_DIR} && python 02b_bertopic_multilingual.py \
    --data_dir {DATA_DIR} --unit speech


2026-04-25 00:13:01.733602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-25 00:13:01.752834: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777075981.774547    9926 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777075981.781621    9926 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777075981.798544    9926 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# =============================================================================
# CELL 11 -- TOPIC COMPARISON: EN translation run (02a) vs Multilingual pooled run (02b)
# =============================================================================
#
# Topic  Count_EN  EN label (02a)                                    ML label (02b)
# ──────────────────────────────────────────────────────────────────────────────────
#    -1     6870   -1_ukraine_europe_union_russia                    -1_european_eu_europe_union
#     0     1155   0_energy_climate_ev_gas                           0_ukraine_russia_putin_russian
#     1      670   1_israel_gaza_palestinian_hamas                   1_budget_european_report_thank
#     2      372   2_ukraine_war_putin_peace                         2_israel_gaza_hamas_palestinian
#     3      369   3_iran_iranian_regime_death                       3_women_gender_violence_equality
#     4      351   4_women_gender_violence_equality                  4_hungary_orbán_law_poland
#     5      301   5_es_digital_dsa_yes                              5_china_hong_kong_hong kong
#     6      285   6_strategic_defence_nato                          6_serbia_kosovo_bosnia_enlargement
#     7      250   7_volt_hungary_orbán_hungarian                    7_farmers_food_agriculture_food security
#     8      243   8_spyware_pegasus_greece_surveillance             8_migration_asylum_pact_migrants
#     9      234   9_china_taiwan_chinese_trade                      9_energy_gas_electricity_prices
#    10      218   10_funding funding_corruption_transparency        10_health_medicines_covid_patients
#    11      211   11_trump_trade_tariffs_united states              11_defence_nato_europe_european
#    12      194   12_culture culture_migration_poland_asylum        12_iran_iranian_regime_women
#    13      183   13_poland_polish_poland_law                       13_trade_trump_united states_tariffs
#    14      181   14_food_russia_food security                      14_syria_turkey_türkiye_syrian
#    15      178   15_belarus_lukashenko_belarusian_political pris.  15_climate_emissions_carbon_climate change
#    16      167   16_children_crimes_ukrainian_war                  16_li_tal_illi_tagƒßna
#    17      162   17_armenia_azerbaijan_karabakh_nagorno            17_energy_gas_russian_russia
#    18      161   18_farmers_agriculture_food_agricultural          18_belarus_lukashenko_belarusian_regime
# ──────────────────────────────────────────────────────────────────────────────────

# We have modified the initial frames script to add a new frames (food security and gender security) given
# large and stable clusters in both english and pooled run.

In [23]:
# ==============================================================================
# CELL 12 — Paragraph-level classification for both ML and EN (as a
# robustness check)
# ==============================================================================
#
# mDeBERTa-v3-mnli-xnli on original-language text (24 EU languages)
# Threshold: 0.4 (set in frames.py)
#
# Following a first run at speech level, the script instead classifies at
# paragraph level (segmented inside the script) rather than speech level.
# Each speech is split on blank lines (fallback: sentence boundaries),
# paragraphs < 60 chars dropped. NLI model sees a focused
# ~100-200 word unit rather than a full 500-word speech.
#
# # RATIONALE FOR ML AS MAIN SPEC:
#
# Of 19,859 speeches, ~92% are delivered in English; the remaining ~1500
# non-English speeches span 21 EU languages, dominated by DE,
# FR, MT (largely Metsola's procedural presidency speeches),
# PL, ES and IT. At least three quarters of the non-English tail is
# in languages well-represented in mDeBERTa's training data.
#
# We use the source-language run as our main specification because:
#
#   1. AUTHENTIC FRAMING. Where MEPs do speak in their own language,
#      that linguistic choice is part of the empirical phenomenon.
#      Translation through an external pipeline mediates this away.
#      The volume is small but the methodological commitment is real.
#
#   2. NO EXTERNAL TRANSLATION DEPENDENCY. The classification depends
#      on a single, characterisable model (mDeBERTa-xnli) rather than
#      mDeBERTa plus an external translation pipeline whose biases are
#      harder to audit and may change without notice.
#
#   3. SINGLE MODEL, ONE FAILURE MODE. Cross-lingual transfer is
#      imperfect on thin-XNLI-data languages (MT, LV, ET, SL, etc.) but
#      this affects <1% of the corpus and is reported as a limitation.
#
# Checkpoint to save progress after each chunk (2000 paragraphs)
#
# Frame shares by year are saved automatically as:
#   frames_classified_para_ML_frame_shares_by_year.csv
#

# Paragraph-level ML
!cd {SCRIPT_DIR} && python 03_classify_frames_zeroshot.py \
    --data_dir {DATA_DIR} \
    --variant ML \
    --unit para


FRAME_THRESHOLD   = 0.4
MIN_PARA_CHARS    = 450
Variant           = ML
Language overrides= True
[skip] Output already exists: /content/drive/MyDrive/classification_pipeline_security_framing/data/frames_classified_para_ML.csv
Delete it to rerun.


In [10]:
# @title

# ==============================================================================
# CELL 13 -- Quick optional inspection at checkpoints to check performance
# on sub-sample
# ==============================================================================

# if frame set-up needs to be tested and modified, this can be run on a
# checkpoint-file with a subset of paragraphs

import os
import pandas as pd
import numpy as np

# --- Load ---
df_ml_raw = pd.read_pickle(os.path.join(DATA_DIR, "_ckpt_classify_para_ML.pkl"))

# --- Reshape ---
df_ml = pd.DataFrame(np.vstack(df_ml_raw["rows"]))

# --- Assign column names (based on your structure) ---
meta_cols = [
    "id", "date", "year", "speaker", "role", "party", "lang", "text"
]

frame_cols = [
    "frame_1","frame_2","frame_3","frame_4","frame_5","frame_6","frame_7","frame_8",
    "frame_9","frame_10","frame_11","frame_12","frame_13","frame_14","frame_15","frame_16"
]

extra_cols = ["top_frame", "top_score", "multi_frames"]

df_ml.columns = meta_cols + frame_cols + extra_cols

# --- TOP-1 ---
print("\n=== TOP-1 (ML) ===")
print((df_ml["top_frame"].value_counts(normalize=True)*100).round(2))


# --- MULTI-LABEL (from your string column) ---
def explode_frames(series):
    return series.dropna().str.split(", ").explode()

print("\n=== MULTI-LABEL (ML) ===")
print((explode_frames(df_ml["multi_frames"]).value_counts(normalize=True)*100).round(2))



=== TOP-1 (ML) ===
top_frame
economic                            35.10
not_security                        20.85
military_defence                    11.40
terrorism                           10.70
border_migration                    10.50
energy                               6.05
health                               3.50
gender_based_violence                0.60
environmental                        0.40
organised_crime                      0.30
foreign_information_interference     0.25
institutional_procedural             0.25
food_security                        0.05
cyber                                0.05
Name: proportion, dtype: float64

=== MULTI-LABEL (ML) ===
multi_frames
economic                            18.48
border_migration                    15.25
terrorism                           14.75
energy                              12.68
military_defence                    11.39
                                     5.94
gender_based_violence                5.80
organised_crime 

In [15]:
# @title
# ==============================================================================
# CELL 14 -- THRESHOLD-SENSITIVE SUMMARY of main specification
# ==============================================================================

# quick inspection, deeper analysis and visualization is done separately

import pandas as pd
import numpy as np
import os

DATA_DIR = "/content/drive/MyDrive/ep_security_framing/data"
FILE = "frames_classified_para_ML.csv"
THRESHOLD = 0.4   # can be increased to get a stricter view

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(os.path.join(DATA_DIR, FILE))

frame_labels = [c.replace("score_", "") for c in df.columns if c.startswith("score_")]
score_cols = [f"score_{f}" for f in frame_labels]

print(f"Loaded: {len(df):,} paragraphs")
print(f"Frames: {len(frame_labels)}")
print(f"Threshold: {THRESHOLD}")

# -----------------------------
# TOP-1 (THRESHOLD-AWARE)
# -----------------------------
scores = df[score_cols].values

top1_idx = scores.argmax(axis=1)
top1_scores = scores[np.arange(len(df)), top1_idx]

top1_labels = [
    frame_labels[i] if top1_scores[r] >= THRESHOLD else "not_security"
    for r, i in enumerate(top1_idx)
]

df["top1_adj"] = top1_labels
df["top1_score_adj"] = top1_scores

# -----------------------------
# SUMMARY
# -----------------------------
print("\n=== TOP-1 DISTRIBUTION (ADJUSTED) ===")
print((df["top1_adj"].value_counts(normalize=True)*100).round(2).to_string())

below = (df["top1_score_adj"] < THRESHOLD).sum()
print("\n=== THRESHOLD DIAGNOSTICS ===")
print(f"Below threshold → not_security: {below:,} ({100*below/len(df):.1f}%)")

print("\n=== TOP-1 SCORE DISTRIBUTION ===")
print(df["top1_score_adj"].describe().round(3).to_string())

# -----------------------------
# MULTI-LABEL
# -----------------------------
scores_mat = scores >= THRESHOLD

n_above = scores_mat.sum(axis=1)

print("\n=== MULTI-LABEL LOAD ===")
print(f"Mean frames per paragraph: {n_above.mean():.2f}")
print(f"0 frames: {(n_above==0).mean()*100:.1f}%")
print(f"1 frame: {(n_above==1).mean()*100:.1f}%")
print(f"2+ frames: {(n_above>=2).mean()*100:.1f}%")

# -----------------------------
# FRAME SHARES (MULTI-LABEL)
# -----------------------------
ml_share = scores_mat.mean(axis=0)
ml_df = pd.Series(ml_share, index=frame_labels).sort_values(ascending=False)

print("\n=== MULTI-LABEL FRAME SHARES ===")
print((ml_df*100).round(2).to_string())

# -----------------------------
# BY YEAR
# -----------------------------
if "year" in df.columns:
    print("\n=== FRAME SHARES BY YEAR (TOP-1, excl. not_security) ===")

    sec = df[df["top1_adj"] != "not_security"]
    year_totals = sec.groupby("year")["top1_adj"].count()

    counts = sec.groupby(["year", "top1_adj"]).size().reset_index(name="n")
    counts["share"] = counts["n"] / counts["year"].map(year_totals)

    pivot = counts.pivot(index="year", columns="top1_adj", values="share").fillna(0)

    print((pivot*100).round(1).to_string())

Loaded: 78,041 paragraphs
Frames: 13
Threshold: 0.4

=== TOP-1 DISTRIBUTION (ADJUSTED) ===
top1_adj
economic                            35.85
not_security                        20.11
military_defence                    12.32
energy                               9.49
terrorism                            9.31
border_migration                     8.72
health                               1.32
gender_based_violence                0.70
foreign_information_interference     0.54
organised_crime                      0.42
environmental                        0.34
food_security                        0.30
cyber                                0.30
institutional_procedural             0.30

=== THRESHOLD DIAGNOSTICS ===
Below threshold → not_security: 15,691 (20.1%)

=== TOP-1 SCORE DISTRIBUTION ===
count    78041.000
mean         0.675
std          0.284
min          0.000
25%          0.473
50%          0.750
75%          0.928
max          1.000

=== MULTI-LABEL LOAD ===
Mean frames per paragr

In [24]:
# ==============================================================================
# CELL 15 — STEP 03: Robustness checks
# ==============================================================================
# As robustness checks, we hold (a) language constant and vary only the unit
# of analysis — speech → paragraph → context_window — so any change in
# results is attributable to granularity, not to translation effects; and (b)
# run the paragraph level classification on the translated corpus.
#
# Hypotheses, threshold (0.4), and the multi-label diagnostics block all
# carry over unchanged from frames.py and the classification script.

# Context-window-level ML
!cd {SCRIPT_DIR} && python 03_classify_frames_zeroshot.py \
    --data_dir {DATA_DIR} --unit context_window --variant ML

# Paragraph-level EN
# EN: same model, translated non-EN text as opposed to source text
!cd {SCRIPT_DIR} && python 03_classify_frames_zeroshot.py \
    --data_dir {DATA_DIR} \
    --variant EN \
    --unit para



FRAME_THRESHOLD   = 0.4
MIN_PARA_CHARS    = 450
Variant           = ML
Language overrides= True

Loading /content/drive/MyDrive/classification_pipeline_security_framing/data/context_windows_ALL_ML.csv...
  285,450 context_window rows loaded
  using pre-segmented context_window units (no splitting needed)
  285,450 context_window units to classify

  device     = CUDA — NVIDIA A100-SXM4-40GB
  dtype      = torch.float16
  batch_size = 85

Loading MoritzLaurer/mDeBERTa-v3-base-mnli-xnli...
Loading weights: 100% 202/202 [00:00<00:00, 755.52it/s, Materializing param=pooler.dense.weight]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Frames: 13

Classifying 285,450 context_w

In [26]:
# ==============================================================================
# STEP 11: ROBUSTNESS CHECKS
# ==============================================================================
#
# PURPOSE
#   Combined robustness section addressing two questions:
#
#   (1) TRANSLATION:
#       Does running on translated English text produce similar findings
#       to running on source-language text?
#       -> ML main spec at paragraph level vs EN at paragraph level.
#
#   (2) GRANULARITY:
#       Are findings stable across the unit of analysis?
#       -> Paragraph vs context-window on the main spec (ML).
#
# APPROACH
#   (a) Top-1 frames
#   (b) Multi-label-aware: going beyond single-label agreement metrics, we also
#   compare yearly trajectories of per-frame multi-label shares — the share of
#   rows in a given year that clear the threshold on a given frame.
#
# OUTPUT
#   - Per-frame correlation table for (a) and (b)
#   - Translation robustness plot
#   - Granularity robustness plot
# ==============================================================================

!cd {SCRIPT_DIR} && python 04_robustness_checks.py \
    --data_dir {DATA_DIR} \
    --threshold 0.4


Using threshold = 0.4

=== ROBUSTNESS TABLE ===
                                  mean_share  ML_vs_EN_multi  para_vs_window_multi  ML_vs_EN_top1  para_vs_window_top1
border_migration                       0.534           0.960                 0.991          0.821                0.976
cyber                                  0.103           0.928                 0.945          0.978                0.976
economic                               0.666           0.722                 0.974          0.519                0.869
energy                                 0.468           0.992                 0.982          0.992                0.978
environmental                          0.024           0.898                 0.856          0.927                0.905
food_security                          0.149           0.981                 0.984          0.957                0.987
foreign_information_interference       0.063           0.760                 0.692          0.956                0.989
